In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:

import yfinance as yf
import pandas as pd
from datetime import datetime, date , timezone
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType
)
from pyspark.sql.functions import col

# ── 1. Config ─────────────────────────────────────────────────
#TICKER_LIST = ["2488.KL"]
# Filter rows where ticker is in a list
sg_ticker_list = ["AMOE.SI",
"AIMA.SI",
"ESRE.SI",
"LIHS.SI",
"STON.SI"]

TICKER_LIST = spark.table("workspace.default.stocks") \
    .select("Symbol") \
    .filter(~col("Symbol").isin(sg_ticker_list)) \
    .toPandas()["Symbol"] \
    .tolist()
TABLE_NAME  = "workspace.default.fiance_dividend_data"

# ── 2. Init Spark ─────────────────────────────────────────────
spark = SparkSession.builder.appName("YFinance_Dividends").getOrCreate()

# ── 3. Extract function ───────────────────────────────────────
def get_dividend_data(symbol: str) -> pd.DataFrame:
    try:
        ticker    = yf.Ticker(symbol)
        dividends = ticker.dividends

        if dividends.empty:
            print(f"[SKIP] {symbol} — no dividend data")
            return pd.DataFrame()

        df = dividends.reset_index()
        df.columns = ["ex_dividend_date", "dividend_per_share"]

        # Normalize timezone → plain date (required for Spark DateType)
        df["ex_dividend_date"] = (
            pd.to_datetime(df["ex_dividend_date"])
            .dt.tz_localize(None)
            .dt.date
        )

        df["ticker"] = symbol
        df["year"]   = pd.to_datetime(df["ex_dividend_date"]).dt.year

        # Enrich from .info
        info = ticker.info
        #df["dividend_rate"]            = info.get("dividendRate",            None)
        #df["dividend_yield"]           = info.get("dividendYield",           None)
        #df["payout_ratio"]             = info.get("payoutRatio",             None)
        #df["five_yr_avg_yield"]        = info.get("fiveYearAvgDividendYield",None)
        #df["last_dividend_value"]      = info.get("lastDividendValue",       None)

        # Next ex-dividend date (Unix timestamp → date)
        #ex_div_ts = info.get("exDividendDate", None)
        #df["next_ex_dividend_date"] = date.fromtimestamp(ex_div_ts) if ex_div_ts else None

        # TTM dividend & yield
        latest_price = ticker.fast_info.get("lastPrice", None)
        one_year_ago = pd.Timestamp.today() - pd.DateOffset(years=1)
        ttm_total    = dividends[
            dividends.index >= one_year_ago.tz_localize(dividends.index.tz)
        ].sum()
        df["ttm_dividend_total"]       = round(ttm_total, 4)
        df["ttm_dividend_yield_pct"]   = (
            round((ttm_total / latest_price) * 100, 4) if latest_price else None
        )
        df["latest_price"]             = latest_price
        df["extracted_at"]             = datetime.now(timezone.utc).date()

        print(f"[OK] {symbol} — {len(df)} records")
        return df

    except Exception as e:
        print(f"[ERROR] {symbol} — {e}")
        return pd.DataFrame()

# ── 4. Collect all tickers ────────────────────────────────────
all_dfs = [get_dividend_data(s) for s in TICKER_LIST]
combined_pd = pd.concat(
    [df for df in all_dfs if not df.empty],
    ignore_index=True
)

# Column order
combined_pd = combined_pd[[
    "ticker", "ex_dividend_date", "year", "dividend_per_share",
    #"dividend_rate", "dividend_yield", "payout_ratio",
    #"five_yr_avg_yield", 
    #"last_dividend_value",
    #"next_ex_dividend_date", 
    "ttm_dividend_total",
    "ttm_dividend_yield_pct", "latest_price", "extracted_at"
]]

# ── 5. Define schema ──────────────────────────────────────────
schema = StructType([
    StructField("ticker",                 StringType(), True),
    StructField("ex_dividend_date",       DateType(),   True),
    StructField("year",                   DoubleType(), True),
    StructField("dividend_per_share",     DoubleType(), True),
   # StructField("dividend_rate",          DoubleType(), True),
   # StructField("dividend_yield",         DoubleType(), True),
   # StructField("payout_ratio",           DoubleType(), True),
   # StructField("five_yr_avg_yield",      DoubleType(), True),
   # StructField("last_dividend_value",    DoubleType(), True),
   # StructField("next_ex_dividend_date",  DateType(),   True),
    StructField("ttm_dividend_total",     DoubleType(), True),
    StructField("ttm_dividend_yield_pct", DoubleType(), True),
    StructField("latest_price",           DoubleType(), True),
    StructField("extracted_at",           DateType(),   True),
])

# ── 6. Convert & save as Delta table ─────────────────────────
spark_df = spark.createDataFrame(combined_pd, schema=schema)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"\n✅ Saved {spark_df.count()} records → {TABLE_NAME}")
spark_df.show(10, truncate=False)

In [0]:
%skip
import yfinance as yf
import pandas as pd
from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType
)
from pyspark.sql.functions import col

# ── 1. Config ─────────────────────────────────────────────────
#TICKER_LIST = ["2488.KL"]
# Filter rows where ticker is in a list
sg_ticker_list = ["AMOE.SI",
"AIMA.SI",
"ESRE.SI",
"LIHS.SI",
"STON.SI"]

TICKER_LIST = spark.table("workspace.default.stocks") \
    .select("Symbol") \
    .filter(~col("Symbol").isin(sg_ticker_list)) \
    .toPandas()["Symbol"] \
    .tolist()
TABLE_NAME  = "workspace.default.fiance_dividend_data"

# ── 2. Init Spark ─────────────────────────────────────────────
spark = SparkSession.builder.appName("YFinance_Dividends").getOrCreate()

# ── 3. Extract function ───────────────────────────────────────
def get_dividend_data(symbol: str) -> pd.DataFrame:
    try:
        ticker    = yf.Ticker(symbol)
        dividends = ticker.dividends

        if dividends.empty:
            print(f"[SKIP] {symbol} — no dividend data")
            return pd.DataFrame()

        df = dividends.reset_index()
        df.columns = ["ex_dividend_date", "dividend_per_share"]

        # Normalize timezone → plain date (required for Spark DateType)
        df["ex_dividend_date"] = (
            pd.to_datetime(df["ex_dividend_date"])
            .dt.tz_localize(None)
            .dt.date
        )

        df["ticker"] = symbol
        df["year"]   = pd.to_datetime(df["ex_dividend_date"]).dt.year

        # Enrich from .info
        info = ticker.info
        df["dividend_rate"]            = info.get("dividendRate",            None)
        df["dividend_yield"]           = info.get("dividendYield",           None)
        df["payout_ratio"]             = info.get("payoutRatio",             None)
        df["five_yr_avg_yield"]        = info.get("fiveYearAvgDividendYield",None)
        df["last_dividend_value"]      = info.get("lastDividendValue",       None)

        # Next ex-dividend date (Unix timestamp → date)
        ex_div_ts = info.get("exDividendDate", None)
        df["next_ex_dividend_date"] = date.fromtimestamp(ex_div_ts) if ex_div_ts else None

        # TTM dividend & yield
        latest_price = ticker.fast_info.get("lastPrice", None)
        one_year_ago = pd.Timestamp.today() - pd.DateOffset(years=1)
        ttm_total    = dividends[
            dividends.index >= one_year_ago.tz_localize(dividends.index.tz)
        ].sum()
        df["ttm_dividend_total"]       = round(ttm_total, 4)
        df["ttm_dividend_yield_pct"]   = (
            round((ttm_total / latest_price) * 100, 4) if latest_price else None
        )
        df["latest_price"]             = latest_price
        df["extracted_at"]             = datetime.utcnow().date()

        print(f"[OK] {symbol} — {len(df)} records")
        return df

    except Exception as e:
        print(f"[ERROR] {symbol} — {e}")
        return pd.DataFrame()

# ── 4. Collect all tickers ────────────────────────────────────
all_dfs = [get_dividend_data(s) for s in TICKER_LIST]
combined_pd = pd.concat(
    [df for df in all_dfs if not df.empty],
    ignore_index=True
)

# Column order
combined_pd = combined_pd[[
    "ticker", "ex_dividend_date", "year", "dividend_per_share",
    "dividend_rate", "dividend_yield", "payout_ratio",
    "five_yr_avg_yield", "last_dividend_value",
    "next_ex_dividend_date", "ttm_dividend_total",
    "ttm_dividend_yield_pct", "latest_price", "extracted_at"
]]

# ── 5. Define schema ──────────────────────────────────────────
schema = StructType([
    StructField("ticker",                 StringType(), True),
    StructField("ex_dividend_date",       DateType(),   True),
    StructField("year",                   DoubleType(), True),
    StructField("dividend_per_share",     DoubleType(), True),
    StructField("dividend_rate",          DoubleType(), True),
    StructField("dividend_yield",         DoubleType(), True),
    StructField("payout_ratio",           DoubleType(), True),
    StructField("five_yr_avg_yield",      DoubleType(), True),
    StructField("last_dividend_value",    DoubleType(), True),
    StructField("next_ex_dividend_date",  DateType(),   True),
    StructField("ttm_dividend_total",     DoubleType(), True),
    StructField("ttm_dividend_yield_pct", DoubleType(), True),
    StructField("latest_price",           DoubleType(), True),
    StructField("extracted_at",           DateType(),   True),
])

# ── 6. Convert & save as Delta table ─────────────────────────
spark_df = spark.createDataFrame(combined_pd, schema=schema)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"\n✅ Saved {spark_df.count()} records → {TABLE_NAME}")
spark_df.show(10, truncate=False)